# RAG Sufficient Context — Kaggle Pipeline

End-to-end selective-generation pipeline (BM25 + LLM + autorater + logistic gate).

**Settings → Accelerator = GPU T4 x1**. Internet: **On**.

## 1. Sanity — GPU + disk

In [ ]:
!nvidia-smi || echo 'no GPU — enable T4 in Settings → Accelerator'
!df -h /kaggle/working | tail -1


## 2. Clone repo

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/SachaYT1/rag-sufficient-context.git"
WORKDIR  = "/kaggle/working/rag-sufficient-context"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORKDIR], check=True)
else:
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=True)

os.chdir(WORKDIR)
print("cwd:", os.getcwd())
!ls


## 3. Install dependencies

Kaggle ships torch, transformers, datasets, numpy, scikit-learn, matplotlib.
We only add what is missing and pin scipy to match the pre-installed numpy ABI
(numpy 1.x → scipy<1.14; numpy 2.x → scipy>=1.14).

In [ ]:
import sys, subprocess
import numpy as np

numpy_major = int(np.__version__.split(".")[0])
print(f"pre-installed numpy {np.__version__}  (major={numpy_major})")

# scipy must match the numpy ABI already loaded:
#   numpy 1.x  ->  scipy < 1.14  (dtype=88 bytes)
#   numpy 2.x  ->  scipy >= 1.14 (dtype=96 bytes)
scipy_pin = "scipy>=1.11,<1.14" if numpy_major < 2 else "scipy>=1.14"

packages = [
    "rank-bm25>=0.2.2",
    "pyyaml>=6.0",
    # accelerate is pre-installed on Kaggle; re-installing it here
    # upgrades to a version incompatible with Kaggle torch and causes
    # "no kernel image available" CUDA errors on T4/P100.
    scipy_pin,
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

import torch, transformers, datasets, sklearn, scipy, matplotlib
from sklearn.linear_model import LogisticRegression

print(f"torch        {torch.__version__}  cuda={torch.cuda.is_available()}")
print(f"transformers {transformers.__version__}")
print(f"datasets     {datasets.__version__}")
print(f"sklearn      {sklearn.__version__}")
print(f"scipy        {scipy.__version__}")
print(f"numpy        {np.__version__}")
print("all imports OK")


## 4. Hugging Face auth (optional)

Only needed for gated models (Llama-3.1-8B). TinyLlama and Mistral-7B-Instruct-v0.3 are open-weights.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("HF_TOKEN")
# from huggingface_hub import login; login(token=token)
print("Skipping HF login")


## 5. Config

Change `model_name` and `num_examples` here.
`TinyLlama/TinyLlama-1.1B-Chat-v1.0` — fast, ~10 min on T4.
`mistralai/Mistral-7B-Instruct-v0.3`  — better quality, ~60 min on T4.

In [ ]:
CONFIG = {
    "execution": {
        "output_dir": "/kaggle/working/results",
        "seed": 42,
    },
    "dataset": {
        "num_examples": 500,  # set to 50 for a quick smoke-test
        "seed": 42,
    },
    "generation": {
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        # "model_name": "mistralai/Mistral-7B-Instruct-v0.3",
        "device_map": "cuda:0",  # explicit GPU — avoids accelerate version mismatch on Kaggle
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "max_new_tokens": 96,
        "temperature": 0.0,
        "top_p": 1.0,
        "top_k": 50,
        "repetition_penalty": 1.0,
    },
    "retrieval": {
        "top_k": 5,
        "max_context_tokens": 4096,
    },
    "autorater": {
        "chunk_size_tokens": 1400,
        "aggregation": "support_all_required",
        "max_new_tokens": 64,
    },
    "confidence": {
        "method": "inline",  # inline | self_report | token_entropy
    },
    "gate": {
        "threshold_steps": 50,
    },
    "evaluation": {
        "f1_threshold": 0.5,
    },
}
print(CONFIG)


## 6. Imports from src

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, WORKDIR)

from src.utils import set_global_seed, save_run_metadata
from src.retrieval import load_hotpotqa, build_retrieval_pipeline, summarize_retrieval_metrics
from src.generation import load_model, generate_answers_batch
from src.autorater import rate_all_examples
from src.confidence import estimate_confidence_batch
from src.evaluation import evaluate_all
from src.gate import (
    compute_selective_curves,
    plot_accuracy_coverage,
    plot_sufficiency_breakdown,
    plot_calibration_curve,
    plot_score_distributions,
    plot_support_recall_vs_f1,
)

set_global_seed(CONFIG["execution"]["seed"])

OUTPUT_DIR = Path(CONFIG["execution"]["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output dir:", OUTPUT_DIR)


## 7. Load HotPotQA (distractor, validation)

In [ ]:
examples = load_hotpotqa(
    split="validation",
    num_examples=CONFIG["dataset"]["num_examples"],
    seed=CONFIG["dataset"]["seed"],
)
print(f"loaded {len(examples)} examples")
print("sample:", examples[0]["question"])


## 8. BM25 Retrieval

In [ ]:
retrieved_examples = build_retrieval_pipeline(
    examples,
    top_k=CONFIG["retrieval"]["top_k"],
    max_context_tokens=CONFIG["retrieval"]["max_context_tokens"],
    tokenizer=None,
)

retrieval_summary = summarize_retrieval_metrics(retrieved_examples)
with open(OUTPUT_DIR / "retrieval_summary.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_summary, f, indent=2)
print(json.dumps(retrieval_summary, indent=2))


## 9. Load model

In [ ]:
model, tokenizer = load_model(model_config=CONFIG["generation"])
print("loaded:", CONFIG["generation"]["model_name"])


## 10. Generate answers

In [ ]:
generated_examples = generate_answers_batch(
    retrieved_examples,
    model, tokenizer,
    max_new_tokens=CONFIG["generation"]["max_new_tokens"],
    temperature=CONFIG["generation"]["temperature"],
    top_p=CONFIG["generation"]["top_p"],
    top_k=CONFIG["generation"]["top_k"],
    repetition_penalty=CONFIG["generation"]["repetition_penalty"],
)

ex0 = generated_examples[0]
print(f"Q:    {ex0['question']}")
print(f"Gold: {ex0['answer']}")
print(f"Pred: {ex0['prediction']}")
print(f"Conf: {ex0['confidence']:.3f}")


## 11. Sufficiency autorater

In [ ]:
rated_examples = rate_all_examples(
    generated_examples,
    model, tokenizer,
    chunk_size_tokens=CONFIG["autorater"]["chunk_size_tokens"],
    aggregation=CONFIG["autorater"]["aggregation"],
    max_new_tokens=CONFIG["autorater"]["max_new_tokens"],
)

n_suf = sum(1 for ex in rated_examples if ex.get("sufficient"))
n_tot = len(rated_examples)
print(f"Sufficient:   {n_suf}/{n_tot} ({n_suf/n_tot:.1%})")
print(f"Insufficient: {n_tot-n_suf}/{n_tot} ({(n_tot-n_suf)/n_tot:.1%})")


## 12. Confidence estimation

In [ ]:
confidence_examples = estimate_confidence_batch(
    rated_examples,
    model=model,
    tokenizer=tokenizer,
    method=CONFIG["confidence"]["method"],
)
print(f"method: {CONFIG['confidence']['method']}")
print("sample:", [round(ex["confidence"], 3) for ex in confidence_examples[:8]])


## 13. Evaluation

In [ ]:
evaluation = evaluate_all(
    confidence_examples,
    f1_threshold=CONFIG["evaluation"]["f1_threshold"],
    output_dir=str(OUTPUT_DIR),
    config=CONFIG,
    model_name=CONFIG["generation"]["model_name"],
)

m = evaluation["metrics"]
print(f"Total:       {m['total']}")
print(f"Correct:     {m['correct']}  ({m['correct_rate']:.1%})")
print(f"Abstain:     {m['abstain']}  ({m['abstain_rate']:.1%})")
print(f"Hallucinate: {m['hallucinate']}  ({m['hallucinate_rate']:.1%})")
print(f"Mean F1:     {m['mean_f1']:.3f}")
print(f"Mean EM:     {m['mean_em']:.3f}")
print(f"Answered accuracy:    {m['answered_accuracy']:.3f}")
print(f"Safe abstention rate: {m['safe_abstention_rate']:.3f}")
print(f"Unsafe answer rate:   {m['unsafe_answer_rate']:.3f}")


## 14. Selective generation gate

In [ ]:
row_by_id = {row["id"]: row for row in evaluation["per_example"]}
examples_for_gate = [
    {**ex, **row_by_id[ex["id"]]}
    for ex in confidence_examples
    if ex["id"] in row_by_id
]

curves = compute_selective_curves(
    examples_for_gate,
    threshold_steps=CONFIG["gate"]["threshold_steps"],
)

with open(OUTPUT_DIR / "selective_curves.json", "w", encoding="utf-8") as f:
    json.dump(curves, f, indent=2, ensure_ascii=False)

print("Gate meta:", json.dumps(curves["gate_meta"], indent=2))
print(f"Answered rows:  {curves['num_answered_rows']}")
print(f"Class balance:  {curves['class_balance_answered']}")
if curves["baseline"].get("aurc") is not None:
    print(f"Baseline AURC:  {curves['baseline']['aurc']:.4f}")
if curves["proposed"].get("aurc") is not None:
    print(f"Proposed AURC:  {curves['proposed']['aurc']:.4f}")
print("Calibration:", json.dumps(curves["calibration"], indent=2))


## 15. Plots

In [ ]:
from IPython.display import Image, display

plot_accuracy_coverage(
    curves, save_path=str(OUTPUT_DIR / "accuracy_coverage.png"))

plot_sufficiency_breakdown(
    examples_for_gate, save_path=str(OUTPUT_DIR / "sufficiency_breakdown.png"))

conf_s = curves.get("confidence_scores", [])
gate_s = curves.get("gate_scores", [])
labels = curves.get("labels", [])

plot_calibration_curve(
    conf_s, labels, title="Confidence-only Calibration",
    save_path=str(OUTPUT_DIR / "calibration_confidence.png"))

plot_calibration_curve(
    gate_s, labels, title="Gate Score Calibration",
    save_path=str(OUTPUT_DIR / "calibration_gate.png"))

plot_score_distributions(
    examples_for_gate, score_key="confidence",
    save_path=str(OUTPUT_DIR / "confidence_distribution.png"))

plot_support_recall_vs_f1(
    evaluation["per_example"],
    save_path=str(OUTPUT_DIR / "support_recall_vs_f1.png"))

for png in sorted(OUTPUT_DIR.glob("*.png")):
    print(png.name)
    display(Image(str(png)))


## 16. Save results + archive for download

In [ ]:
import shutil, datetime

with open(OUTPUT_DIR / "examples_enriched.json", "w", encoding="utf-8") as f:
    json.dump(examples_for_gate, f, indent=2, ensure_ascii=False)

save_run_metadata(
    output_dir=str(OUTPUT_DIR),
    config=CONFIG,
    model_name=CONFIG["generation"]["model_name"],
    extra={
        "num_examples": m["total"],
        "correct": m["correct"],
        "abstain": m["abstain"],
        "hallucinate": m["hallucinate"],
    },
)

artifacts = sorted(p.name for p in OUTPUT_DIR.iterdir())
print("Artifacts:", artifacts)

stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
archive = f"/kaggle/working/rag_results_{stamp}"
shutil.make_archive(archive, "zip", str(OUTPUT_DIR.parent), OUTPUT_DIR.name)
print("Archive:", archive + ".zip")
!ls -lh /kaggle/working/*.zip
